# Employee Attrition Simulation (JPSED)

労働者の属性から**翌年の離職**を予測する。母集団は**雇用者のみ**。目的変数は2種を構築して比較する:

- **実離職 `attrition_separation`**: 翌年調査で「仕事を辞めた・退職した」(Q57_1/Q58_1) と申告 → 実際の離職行動 (基準率~8%)。
- **転職・就職意向 `attrition_intention`**: 当年の Q106/Q105 が積極的意向 (コード1-2) → 意思 (基準率~19%)。

Colab / ローカルどちらでも動く。`DATA_MODE` でデータソースを切替。

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# 'synthetic' : 同梱の合成データ（申請不要）
# 'gdrive'    : Google Drive 上の JPSED データ（Colab）
# 'local'     : ローカルの JPSED データ
DATA_MODE = 'synthetic'

GDRIVE_DATA_DIR = '/content/drive/MyDrive/ColabNotebooks/employee-attrition/data'
LOCAL_DATA_DIR  = '/Users/yoshikawadaisuke/My Drive/ColabNotebooks/employee-attrition/data'
REPO_URL        = 'https://github.com/dsk-yshkw/employee-attrition'

In [ ]:
# ── Install / clone repo (Colab) ───────────────────────────────────────────────
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('employee-attrition'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('employee-attrition')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
    if DATA_MODE == 'gdrive':
        from google.colab import drive
        drive.mount('/content/drive')

# repo ルートを import パスに追加（ローカル実行で notebooks/ から開いた場合に対応）
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [ ]:
# ── Resolve data directory ─────────────────────────────────────────────────────
SYNTHETIC = (DATA_MODE == 'synthetic')
if SYNTHETIC:
    DATA_DIR = 'data/synthetic'
    # 合成の年次CSVが無ければ生成
    if not os.path.exists('data/synthetic/2022.csv'):
        subprocess.run([sys.executable, 'data/synthetic/generate_synthetic.py'], check=True)
elif DATA_MODE == 'gdrive':
    DATA_DIR = GDRIVE_DATA_DIR
else:
    DATA_DIR = LOCAL_DATA_DIR
print('DATA_MODE :', DATA_MODE)
print('DATA_DIR  :', DATA_DIR)

## 1. パネル構築とラベル（母集団=雇用者）

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data.loader import DataLoader
from src.data.panel_builder import PanelBuilder
from src.features.assembler import FeatureAssembler
from src.config import TARGET_SEPARATION, TARGET_INTENTION

loader = DataLoader(DATA_DIR, synthetic=SYNTHETIC)
pb = PanelBuilder(loader)
fa = FeatureAssembler()

panel = pb.add_targets(pb.build())
emp = pb.employee_mask(panel)
sub = panel[emp]
print('person-year rows:', len(panel), '| employees:', int(emp.sum()),
      '| persons:', panel['pkey'].nunique())
print('陽性率  separation=%.3f  intention=%.3f' %
      (sub[TARGET_SEPARATION].dropna().mean(), sub[TARGET_INTENTION].dropna().mean()))
sub.groupby('year')[[TARGET_SEPARATION, TARGET_INTENTION]].mean().round(3)

## 1b. 名目 vs 実質年収の推移（インフレによる実質賃金の目減り）

JPSEDのwave Yは前年(Y-1)の年収を尋ねるため、CPIはY-1年で実質化。2022–2024で名目は上昇するが実質は下落する。

In [ ]:
from src.data.macro import MacroFeatureBuilder
X_macro = MacroFeatureBuilder().build(sub)
tmp = sub.assign(nominal=X_macro['real_annual_income']*0 + sub['annual_income'].astype(float),
                 real=X_macro['real_annual_income'], cpi=X_macro['cpi'])
g = tmp.groupby('year')[['nominal','real','cpi']].mean()
print(g.round(1).to_string())
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(g.index, g['nominal'], 'o-', label='Nominal income (man-yen)')
ax.plot(g.index, g['real'],    's-', label='Real income (2020 yen)')
ax.set_xlabel('survey year'); ax.set_ylabel('mean annual income'); ax.legend(); ax.set_title('Nominal vs Real income')
plt.tight_layout(); plt.show()

## 2. モデル学習・評価（時系列split, 2ターゲット × 複数モデル）

In [ ]:
from src.pipeline import run_experiment, _xgboost_available
models = ['logistic', 'histgbm'] + (['xgboost'] if _xgboost_available() else [])
plans  = [(TARGET_SEPARATION, 2024, 'Separation'), (TARGET_INTENTION, 2025, 'Intention')]

rows, results = [], {}
for target, test_year, name in plans:
    for m in models:
        r = run_experiment(pb, fa, target, test_year, model_type=m)
        results[(target, m)] = r
        rows.append(dict(target=name, model=m, test_year=test_year,
                         n_train=r.n_train, n_test=r.n_test,
                         pos_rate=round(r.test_pos_rate, 3),
                         AUC=round(r.metrics['auc_roc'], 3),
                         PR_AUC=round(r.metrics['auc_pr'], 3)))
summary = pd.DataFrame(rows)
best = {t: max(models, key=lambda m: results[(t, m)].metrics['auc_roc']) for t, _, _ in plans}
summary

## 3. ROC / PR 曲線（各ターゲットのベストモデル）

In [ ]:
from src.models.evaluator import ModelEvaluator
ev = ModelEvaluator()
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, (target, test_year, name) in enumerate(plans):
    fr = pb.build_modeling_frame(target)
    te = fr[fr['year'] == test_year]
    yp = results[(target, best[target])].model.predict_proba(fa.build(te))
    ev.plot_roc(te[target], yp, ax=axes[i, 0]); axes[i, 0].set_title(f'{name} ROC ({best[target]})')
    ev.plot_pr(te[target], yp, ax=axes[i, 1]);  axes[i, 1].set_title(f'{name} PR ({best[target]})')
plt.tight_layout(); plt.show()

## 4. 特徴量重要度（Permutation importance）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (target, test_year, name) in zip(axes, plans):
    imp = results[(target, best[target])].importance.sort_values().tail(12)
    imp.plot(kind='barh', ax=ax, color='#55A868')
    ax.set_title(f'{name}: {best[target]} 重要度 top12'); ax.set_xlabel('AUC drop')
plt.tight_layout(); plt.show()

## 5. 考察

- **実離職**の主要予測子は勤続年数(tenure)・年齢・年収・雇用形態。
- **意向**は雇用形態・年齢の寄与が相対的に大きい。
- **年収増加率**の寄与は小さい。定期昇給と昇進/転職/労働時間変化が混在した粗い指標（JPSEDに基本給と定期昇給の内訳が無い）。
- 実離職は稀事象(~8%)でPR-AUCが低め。実務では閾値調整・class weight・コスト考慮が重要。

> 実データは SSJDA 利用許諾のため非公開。`DATA_MODE` で合成/実データを切替可能。